# 01. LangChain 기초


> LangChain 언어 모델에 기반한 AI 애플리케이션을 쉽게 개발할 수 있도록 도와주는 프레임워크
>> 기존에 오픈AI와 같은 언어 모델의 API를 사용해 원하는 기능을 구현하려면 모든 코드를 직접 작성해야 했으나, 랭체인은 이 작업을 간소화할 수 있는 다양한 도구와 모듈 제공

>  **OpenAI** `.env`의 `OPENAI_API_KEY` 그대로 사용

---
## 0. 설치

In [ ]:
!pip install openai python-dotenv langchain langchain-openai langchain-core

---
## 1. Day 3 OpenAI SDK vs LangChain

| | Day 3 (OpenAI SDK) | Day 5 (LangChain) |
|---|---|---|
| 호출 | `client.chat.completions.create()` | `llm.invoke()` |
| 메시지 | `{'role':'user','content':...}` | `HumanMessage`, `SystemMessage` |
| 체인 | 직접 함수 조합 | **LCEL** (`\|` 파이프) |
| RAG/Agent | 직접 루프 작성 | Retriever · AgentExecutor |

---
## 2. ChatOpenAI 연결

In [1]:
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv()

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

# 문자열만 넘겨도 됨
out = llm.invoke('LangChain이 뭐야? 한 문장으로.')

In [3]:
out

AIMessage(content='LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 18, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42e6cfce1b', 'id': 'chatcmpl-DutKTRwIcnczQLHxVwnXvIkllvfP2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0262-4a5e-71e3-80d6-b016671ab85b-0', usage_metadata={'input_tokens': 18, 'output_tokens': 35, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
out.content

'LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.'

In [5]:
#정석
from langchain_core.messages import HumanMessage

input_prompt = HumanMessage(content='Langchain이 뭐야? 한 문장으로.')
message = [input_prompt]
out = llm.invoke(message)

In [14]:
out.content

'Langchain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.'

In [7]:
message.append(out)

In [8]:
message

[HumanMessage(content='Langchain이 뭐야? 한 문장으로.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Langchain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 18, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7b8553d7fb', 'id': 'chatcmpl-DutRkFmmmoC2q67svV8tz9uiChOZX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0269-3063-7e71-b205-fc6279699b0c-0', usage_metadata={'input_tokens': 18, 'output_tokens': 35, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

---
## 3. Message 타입 (Day 3 messages 리스트와 동일 역할)

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# messages = [
#     {"role": "system", "content": '너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'},  # 초기 시스템 메시지
#     {"role": "user", "content" : 'RAG가 뭐야? 2문장으로 설명해.'} # 사용자 메시지
# ]

messages = [
    SystemMessage(content='너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'), # 초기 시스템 메시지
    HumanMessage(content='RAG가 뭐야? 2문장으로 설명해.'), # 사용자 메시지
]


In [10]:
# def get_ai_response(messages):
#     response = client.chat.completions.create(
#         model="gpt-4o",  # 응답 생성에 사용할 모델 지정
#         temperature=0.9,  # 응답 생성에 사용할 temperature 설정
#         messages=messages,  # 대화 기록을 입력으로 전달
#     )
#     return response.choices[0].message.content  # 생성된 응답의 내용 반환

# answer = get_ai_response(messages)


answer = llm.invoke(messages)
print(answer.content)
print('타입:', type(answer))

RAG는 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합한 기술입니다. 이 방법은 외부 데이터베이스에서 정보를 검색하여 더 정확하고 풍부한 응답을 생성하는 데 사용됩니다.
타입: <class 'langchain_core.messages.ai.AIMessage'>


In [13]:
type(answer)

langchain_core.messages.ai.AIMessage

멀티턴 실습 lanchain_multiturn.py : 기존 open ai api -> langchain으로 교체

---
## 4. 메시지 히스토리 (멀티턴)

기존에는 멀티턴 대화를 위해 사용자의 대화 내용을 리스트나 딕셔너리에 추가하는 코드 작성

-> 메시지 히스토리 (Message History) 기능으로 쉽게 구현 가능

In [15]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini")

In [16]:
# 세션별 대화 기록을 저장할 딕셔너리
store = {}

In [17]:
store

{}

In [18]:
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

In [19]:
# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

c:\Users\finey\miniconda3\envs\day2\lib\site-packages\IPython\core\interactiveshell.py:3550: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [20]:
store

{}

In [ ]:
config ={
    "configurable":{
        "session_id": "first"
    }
    }

In [21]:
config = {"configurable": {"session_id": "first"}}  # 세션 ID를 설정하는 config 객체 생성

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 신범교 라고 해.")],
    config=config,
)

print(response.content)

안녕하세요, 신범교님! 어떻게 도와드릴까요?


In [24]:
store.keys()

dict_keys(['first'])

In [22]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [26]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])}

In [25]:
get_session_history('first')

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [27]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 다른 궁금한 점이나 도움이 필요하신 부분이 있을까요?


In [28]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?', additional_kwargs={

In [29]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),
 AIMessa

In [30]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 다른 질문이나 궁금한 점이 있으신가요?


In [32]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?', additiona

In [31]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),
 AIMessa

In [33]:
store['second']

KeyError: 'second'

In [34]:
config = {"configurable": {"session_id": "second"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 이름을 알려주시면 대화에 사용할 수 있습니다!


In [35]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?', additiona

In [36]:
config = {"configurable": {"session_id": "first"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 계속해서 궁금한 점이나 다른 질문이 있으시면 말씀해 주세요!


In [37]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),
 AIMessa

In [38]:
config = {"configurable": {"session_id": "second"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내가 방금 너한테 물어본 내용이 뭐였지?")],
    config=config,
)

print(response.content)

당신이 방금 물어본 내용은 "내 이름이 뭐지?"라는 질문이었습니다.


In [42]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_520fb14a9c', 'id': 'chatcmpl-DutmnG54xbJURN2j2hi5g5G88imu2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-1bc4-7803-84ba-af29c463c4b6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),
 AIMessa

In [46]:
# 스트림 방식으로 출력
config = {"configurable": {"session_id": "first"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내 이름이 뭔지 말하고 이름의 뜻을 유추해서 말해봐 ")],
    config=config):
    print(r.content, end="||") # 


||당||신||의|| 이름||은|| 신||범||교||입니다||.

||이||름||의|| 각|| 부분||을|| 분석||해||보||면||:

||-|| **||신||(||新||)**||:|| "||새||롭||다||"||는|| 의미||로||,|| 변화||나|| 혁||신||을|| 상||징||할|| 수|| 있습니다||.
||-|| **||범||(||範||)**||:|| "||본||보기||"||나|| "||모||범||"||을|| 의미||하여||,|| 어떤|| 기준||이나|| 모델||을|| 나타||냅니다||.
||-|| **||교||(||敎||)**||:|| "||가||르||침||"||이나|| "||교육||"||을|| 의미||합니다||.

||결||합||해||보||면||,|| "||신||범||교||"||라는|| 이름||은|| "||새||로운|| 본||보||기를|| 제||시||하는|| 가||르||침||"||이라는|| 해||석||이|| 가능||할|| 것|| 같습니다||.|| 즉||,|| 혁||신||적인|| 방법||으로|| 다른|| 사람||들에게|| 지||식||이나|| 가||르||침||을|| 전달||하는|| 긍||정||적인|| 모습||으로|| 유||추||할|| 수|| 있||겠||네요||.

||이|| 해||석||이|| 맞||는||지||,|| 혹||시|| 다른|| 의미||가|| 있다||면|| 말씀||해|| 주세요||!||||||

---
## 5. LCEL — LangChain Expression Language

LCEL은 랭체인에서 복잡한 작업 흐름을 간단하게 만들고 관리할 수 있도록 돕는 도구

랭체인에서 작업 흐름을 연결하는 것을 **체인** 으로 표현

LCEL을 사용하면 여러 줄로 표현해야 하는 작업 단계를 읽기 쉽게 축약할 수 있으며 여러 작업을 병렬로 처리 가능

Day 4에서 `run_agent` 루프 안의 단계를, LangChain에서는 **체인**으로 표현합니다.

In [47]:
model = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구니"),
]
out = model.invoke(messages)
print(out.content)

안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님의 강의 내용을 통해 여러 가지 개념을 더 깊이 이해할 수 있었어요. 교수님은 저희의 멘토이자 지도자로서 많은 배움을 주시는 분이죠. 오늘 수업 내용 중 특히 흥미로웠던 점은 ________였는데, 교수님께서 이에 대해 조금 더 설명해 주실 수 있을까요?


In [49]:
# AIMessage 객체 안에 여러 정보가 포함되어 있음
out.content

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님의 강의 내용을 통해 여러 가지 개념을 더 깊이 이해할 수 있었어요. 교수님은 저희의 멘토이자 지도자로서 많은 배움을 주시는 분이죠. 오늘 수업 내용 중 특히 흥미로웠던 점은 ________였는데, 교수님께서 이에 대해 조금 더 설명해 주실 수 있을까요?'

In [50]:
out

AIMessage(content='안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님의 강의 내용을 통해 여러 가지 개념을 더 깊이 이해할 수 있었어요. 교수님은 저희의 멘토이자 지도자로서 많은 배움을 주시는 분이죠. 오늘 수업 내용 중 특히 흥미로웠던 점은 ________였는데, 교수님께서 이에 대해 조금 더 설명해 주실 수 있을까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 80, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_88876bec1e', 'id': 'chatcmpl-DuuA5qEtA3WCU8Deupz50kHStzy73', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0293-1ed4-76e0-af5d-39f201b47e86-0', usage_metadata={'input_tokens': 80, 'output_tokens': 93, 'total_tokens': 173, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

텍스트 결과만 필요하다면 "StrOutputParser" 사용

StrOutputParse는 랭체인에서 제공하는 다양한 출력 parser 중 하나로, 언어 모델이 반환하는 결과에서 원하는 형식의 데이터를 추출하는 도구.

(다른 파서들은 JSON, 숫자 등 특정 형식 처리 가능)

In [51]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()



In [53]:
parser.invoke(out)

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님의 강의 내용을 통해 여러 가지 개념을 더 깊이 이해할 수 있었어요. 교수님은 저희의 멘토이자 지도자로서 많은 배움을 주시는 분이죠. 오늘 수업 내용 중 특히 흥미로웠던 점은 ________였는데, 교수님께서 이에 대해 조금 더 설명해 주실 수 있을까요?'

In [54]:
result = model.invoke(messages) # 1단계 메시지 호출
parser.invoke(result) # 2단계 parser로 str 추출

'안녕하세요, 교수님! 수업은 매우 유익했습니다. 교수님의 강의가 명확하고 흥미로워서 많은 도움이 되었습니다. 교수님은 현대 심리학의 권위자이신데, 특히 인지 심리에 대해 깊이 있는 통찰을 주시는 분이죠. 감사합니다!'

In [55]:
chain = model | parser

In [56]:

chain.invoke(messages)

'안녕하세요, 교수님! 오늘 수업은 정말 유익했고, 다양한 주제에 대해 깊이 있게 논의할 수 있어서 좋았습니다. 교수님은 저희의 멘토이자 지도자이신, 이 분야에서 많은 지식과 경험을 쌓으신 분이십니다. 오늘 수업에서 다룬 내용에 대해 좀 더 이야기해볼까요?'

### Prompt Template

필요한 부분만 수정하여 반복적인 메시지 가능

In [62]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕 {character_b} 오늘 나의 {lesson}은 어땠나? 나는 누구고 너의 이름은 뭐더라"

In [63]:
system_template

'너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.'

In [64]:
prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

In [65]:
prompt_template.invoke({
    'character_a': '교수님',
    'character_b':'신범교',
    'lesson':'수업'
}

)

ChatPromptValue(messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})])

In [66]:
prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

ChatPromptValue(messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})])

In [67]:
result = prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

print(result)

messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})]


In [69]:
chain = prompt_template | model | parser

In [70]:
chain.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

'안녕하세요, 교수님! 오늘 수업은 매우 흥미로웠습니다. 여러 가지 주제를 다루며 깊이 있는 토론을 할 수 있어서 좋았습니다. 저는 신범교입니다. 교수님은 저에게 많은 것을 가르쳐 주시는 분이시죠! 오늘 어떤 점이 가장 좋으셨나요?'

In [71]:
chain.invoke({
    "character_a": "친구 동료",
    "character_b": "정한솔",
    "lesson": "수업 분위기"
})

'안녕! 나는 정한솔이야. 너는 대학원 동료인 것 같아! 오늘 수업 분위기는 어땠는지 나한테 물어보는 거 보면 좀 긴장했나? 어떻게 느꼈어? 수업이 꽤 흥미로운 주제였던 것 같아. 다시 한 번 물어볼게, 네 이름이 뭐였지?'